In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from black_scholes.pinn.bs_pinn_nd import BSProductPINN, BSMaxPINN, BSMinPINN
from utility.model import EarlyStopping
from config.bs_nd import *


In [2]:
def train_product_pinns(seeds):
    for seed in seeds:
        print(f'Training model with seed {seed}...')
        pinn = BSProductPINN(model_config, seed=seed)
        pinn.set_params(K, r, sigmas, corr, T, S_mins, S_maxs)
        pinn.set_loss_weights(loss_weights)

        early_stopping = EarlyStopping(patience=500, min_delta=1e-7)
        pinn.train(batch_size=4096, epochs=40000, early_stopping=early_stopping)
        pinn.save(f'../../models/bs_pinn_nd/{seed}.pth')

def train_min_max_pinns(seeds):
    sigmas_ = sigmas
    for seed in seeds:
        sigmas_ += 0.1
        print(f'Training model with seed {seed}...')
        print("sigmas:", sigmas)

        print("Training minimum PINN...")
        pinn_min = BSMinPINN(model_config, seed=seed)
        pinn_min.set_params(K, r, sigmas_, corr, T, S_mins, S_maxs)
        pinn_min.set_loss_weights(loss_weights)
        early_stopping = EarlyStopping(patience=500, min_delta=1e-7)
        pinn_min.train(batch_size=4096, epochs=40000, early_stopping=early_stopping, anneal_freq=50000, alpha=0.9)
        pinn_min.save(f'../../models/bs_pinn_nd/minimum/{seed}.pth')

        print("Training maximum PINN...")
        pinn_max = BSMaxPINN(model_config, seed=seed)
        pinn_max.set_params(K, r, sigmas_, corr, T, S_mins, S_maxs)
        pinn_max.set_loss_weights(loss_weights)
        early_stopping = EarlyStopping(patience=500, min_delta=1e-7)
        pinn_max.train(batch_size=4096, epochs=40000, early_stopping=early_stopping, anneal_freq=50000, alpha=0.9)
        pinn_max.save(f'../../models/bs_pinn_nd/maximum/{seed}.pth')

In [3]:
seeds = range(100, 105)
train_min_max_pinns(seeds)

Training model with seed 100...
sigmas: [0.3 0.4]
Training minimum PINN...
Precomputing boundary interpolation grids...
Done.
Iter      0 | Train: 1.4452e+00 | Val: 5.1863e+00 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter    500 | Train: 1.1723e-02 | Val: 4.8445e-02 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   1000 | Train: 3.2683e-03 | Val: 1.3124e-02 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   1500 | Train: 2.6420e-03 | Val: 1.0209e-02 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   2000 | Train: 1.6030e-03 | Val: 6.2485e-03 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   2500 | Train: 7.8939e-04 | Val: 3.1083e-03 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   3000 | Train: 4.8052e-04 | Val: 1.9644e-03 | Weights: variational=0.250  terminal=0.250  Smax=0.250  Smin=0.250
Iter   3500 | Train: 3.4650e-04 | Val: 1.

KeyboardInterrupt: 